# 🎙️ Arabic Speech Corpus Pipeline — Student 1 Production Studio (v10)
### Project: Arabic Speech Dataset Creation (Prof. Mohsen Rashwan Course)
**Student 1 Deliverable: Steps 1–4 (Ingestion, Multi-Class Labelling, Continuous Energy-Assisted Silence Overlay & Intelligent Segmentation)**

---

### 🌟 Major Upgrades in Version 10:
1. **Continuous Instant-by-Instant Voice Activity Overlay (Short-Time RMS Energy Assisted):**
   - **No block-by-block rounding:** Operates at 10ms frame resolution (100 frames/sec) instead of coarse 1-second blocks.
   - **Energy-Assisted:** Computes Short-Time RMS Log Energy $E_{\text{dB}}(t)$ and an adaptive silence threshold to distinguish real speech oscillations from quiet pauses.
   - **Exact Millisecond Shading:** Waveform shading directly tracks acoustic amplitude bursts (green for `SPEECH`) and quiet intervals (grey for `SILENCE`) with sub-second accuracy.
   - **Dual-Panel Inspector:** Shows the Audio Waveform and the Continuous Energy Curve (dB) with the Silence Threshold line and a movable cursor pointer.
2. **Streamlined Student 1 Scope (Removed Clean/Noisy):**
   - Clean/Noisy acoustic SNR classification has been handed off to **Student 2**, keeping Student 1 focused 100% on slicing, boundary accuracy, silence margins, and content classes (`speech`, `music`, `mixed`, `silence`).
3. **Tiered Cut Engine (0 Forced Blind Cuts):**
   - Tier 1: Safe pause $\ge 250$ms within target window (8–12s).
   - Tier 2a: Extended safe pause (up to 18s).
   - Tier 2b: Lower-threshold pause (120–250ms).
   - Tier 3: `energy_valley_cut` (quietest smoothed RMS energy valley snapped to zero-crossing).
   - Hann Fading: 8ms soft fade-in/fade-out on every cut slice to eliminate acoustic pops and clicks.
4. **Strict Course-Compliant Silence Margins (100–200ms):**
   - Cuts placed at pause midpoints and capped: `seg_end = min(cut_t, pause.start + 0.20s)`, `next_start = max(cut_t, pause.end - 0.20s)`.
   - Guaranteed $100\text{ms} \le \text{margin} \le 200\text{ms}$ on both edges.
5. **Calibrated Urgency & Active Learning Review Queue:**
   - Multi-factor formula: $0.40 \cdot w_{\text{cut}} + 0.25 \cdot \text{disagreement} + 0.20 \cdot (1 - \text{margin}) + 0.15 \cdot w_{\text{speech}}$.
   - Calibrated so **URGENT** segments represent exactly **~10% (5–15% target)** of corpus.


In [ ]:
import sys
from pathlib import Path

# Add repository root to path
REPO_PATH = Path('Audio-Datasets-Creation-System')
if REPO_PATH.exists() and str(REPO_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_PATH.resolve()))

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Audio, HTML

from speech_corpus.audio import read_wav, write_wav, extract_audio, extract_audio_hq, DEFAULT_SR
from speech_corpus.downloader import download_video, find_cached_video
from speech_corpus.labelling import classify_audio, build_mock_audio
from speech_corpus.segmentation import (
    cut_speech_and_mixed_segments,
    detect_speech_activity_overlay,
    compute_instantaneous_energy_vad_overlay,
    create_second_by_second_overlay,
    log_human_change
)
from speech_corpus.inspector import (
    launch_region_inspector,
    launch_vad_overlay_inspector,
    launch_timeline_pointer_inspector,
    launch_segment_inspector
)
from speech_corpus.quality_control import (
    validate_segments_integrity,
    split_corpus_by_source,
    generate_qc_markdown_report
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Environment Ready! • Device: {device.upper()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


## 📥 Steps 1 & 2: YouTube Ingestion & 16 kHz Audio Extraction
- Downloads video at fast resolution (360p / 480p) to preserve bandwidth.
- Checks local disk cache with metadata preservation (`meta.json` / `info.json`).
- Extracts standard 16 kHz mono WAV for the pipeline and archival HQ FLAC.


In [ ]:
USE_MOCK_DEMO = False  # Set to True for an instant offline demo
YOUTUBE_URL = "https://www.youtube.com/watch?v=7fIN_zj4Yno"  # Replace with any YouTube URL
MAX_HEIGHT = 360  # Fast download (360p or 480p)
DATA_DIR = Path("data")

if USE_MOCK_DEMO:
    print("⚡ Generating synthetic test audio (speech, music, mixed, silence)...")
    audio_samples, gt_df = build_mock_audio(seed=42)
    video_dir = DATA_DIR / "mock_demo"
    video_dir.mkdir(parents=True, exist_ok=True)
    wav_path = video_dir / "audio.wav"
    write_wav(wav_path, audio_samples, sample_rate=DEFAULT_SR)
    print(f"✅ Mock audio saved: {wav_path}")
else:
    print(f"📥 Checking/Downloading {YOUTUBE_URL} at {MAX_HEIGHT}p...")
    raw_vid, meta = download_video(YOUTUBE_URL, DATA_DIR / "_incoming", max_height=MAX_HEIGHT)
    video_dir = DATA_DIR / meta["video_id"]
    video_dir.mkdir(parents=True, exist_ok=True)
    final_vid = video_dir / raw_vid.name
    if raw_vid.resolve() != final_vid.resolve() and raw_vid.exists():
        raw_vid.replace(final_vid)

    wav_path = video_dir / "audio.wav"
    flac_path = video_dir / "audio_hq.flac"
    extract_audio(final_vid, wav_path)
    extract_audio_hq(final_vid, flac_path)
    audio_samples, sr = read_wav(wav_path)
    print(f"✅ Video Ready: {meta.get('title', meta['video_id'])}")
    print(f"   • Duration: {len(audio_samples) / DEFAULT_SR / 60.0:.2f} minutes ({len(audio_samples) / DEFAULT_SR:.1f}s)")
    print(f"   • Channel: {meta.get('channel', 'Unknown')} | License: {meta.get('license', 'Standard')}")
    print(f"   • Audio WAV: {wav_path}")


## 🧠 Step 3: AST Multi-Class Classification & Urgency Scoring
- Classifies audio into 4 content classes: `speech`, `music`, `mixed`, `silence`.
- Calculates margin-based Certainty and Urgency.
- **Smart Cache:** If `regions.csv` exists, loads instantly. Set `FORCE_RECLASSIFY = True` to re-run.


In [ ]:
regions_csv = video_dir / "regions.csv"
windows_csv = video_dir / "windows_5s.csv"
FORCE_RECLASSIFY = False  # Set to True to re-run neural inference from scratch

if not FORCE_RECLASSIFY and regions_csv.exists() and windows_csv.exists():
    print(f"⚡ Loading cached AST classification from {regions_csv}...")
    regions_df = pd.read_csv(regions_csv)
    raw_preds = pd.read_csv(windows_csv)
else:
    print("🧠 Running AST classification on GPU (with live batch progress)...")
    regions_df, raw_preds = classify_audio(audio_samples, sr=DEFAULT_SR, backend="ast", device=device)
    regions_df.to_csv(regions_csv, index=False)
    if isinstance(raw_preds, pd.DataFrame):
        raw_preds.to_csv(windows_csv, index=False)

print(f"✅ Identified {len(regions_df)} continuous macro-regions.")
print()
print("Macro-Region Breakdown:")
print(regions_df["label"].value_counts())
print()
print("Review Priority Breakdown:")
print(regions_df["review_priority"].value_counts())

display(regions_df[["start", "end", "duration", "label", "confidence", "certainty", "urgency", "review_priority"]].head(15))


## 📊 Full Audio Spectrogram Overview
Generates and displays the macro spectrogram across the full recording duration.


In [ ]:
spec_path = video_dir / "full_spectrogram.png"
if not spec_path.exists():
    try:
        from speech_corpus.visualization import plot_audio_timeline
        print("Generating timeline visual report...")
        plot_audio_timeline(audio_samples, regions_df, out_path=spec_path, sr=DEFAULT_SR)
    except (ImportError, AttributeError):
        from speech_corpus.visualization import plot_spectrogram_regions
        print("Generating timeline visual report via plot_spectrogram_regions...")
        plot_spectrogram_regions(audio_samples, regions_df, sr=DEFAULT_SR, save_path=spec_path)

from IPython.display import Image
if spec_path.exists():
    display(Image(filename=str(spec_path)))


### 🔍 Interactive Inspector 1: AST Continuous Macro-Regions
Select any macro-region to listen, view its spectrogram, and apply human overrides. All changes are logged to `changes.csv`.


In [ ]:
# 🔍 Cell 1 of 4: Inspect AST Continuous Regions & Human Override
launch_region_inspector(
    audio_samples=audio_samples,
    regions_df=regions_df,
    sr=DEFAULT_SR,
    csv_path=video_dir / "regions.csv"
)


## 🎙️ Step 3.5: Continuous Energy-Assisted Silence Overlay (`SPEECH` vs `SILENCE`)
Unlike macro-region labelling (5–15s chunks), this overlay evaluates audio at **millisecond-level resolution (10ms frames)**:
- **Energy-Assisted:** Uses Short-Time RMS Log Energy $E_{\text{dB}}(t)$ to detect physical vocal cords vibrations vs acoustic silence.
- **`SPEECH` (NO SILENCE):** Acoustic waveform oscillations above silence threshold.
- **`SILENCE` (NO SPEECH):** Natural inter-phrase quiet gaps near the noise floor.
- **Millisecond Precision:** Continuous runs match the exact waveform bursts without coarse 1-second approximations.


In [ ]:
# 🎙️ Step 3.5: Continuous Energy-Assisted Speech vs Silence Overlay
SILENCE_THRESHOLD_S = 0.25  # 250ms threshold: minimum pause duration for a 100% safe cut

print("🔍 Extracting millisecond-level Voice Activity & Short-Time Energy...")
vad_timeline_df, pauses_df = detect_speech_activity_overlay(
    audio=audio_samples,
    sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    min_pause_ms=120,
    vad_backend="auto"
)

# Compute continuous 10ms frame energy overlay
instant_intervals_df, times_arr, energy_db_arr, thresh_db = compute_instantaneous_energy_vad_overlay(
    audio=audio_samples,
    sr=DEFAULT_SR,
    vad_timeline_df=vad_timeline_df
)

total_dur = len(audio_samples) / DEFAULT_SR
sp_time = instant_intervals_df[instant_intervals_df["state"] == "SPEECH"]["duration"].sum() if not instant_intervals_df.empty else 0.0
sil_time = instant_intervals_df[instant_intervals_df["state"] == "SILENCE"]["duration"].sum() if not instant_intervals_df.empty else 0.0

print()
print("📊 Continuous Acoustic Overlay Summary:")
print(f"• Total Audio Duration : {total_dur/60:.1f} mins ({total_dur:.1f}s)")
print(f"• Active Speech Energy : {sp_time/60:.1f} mins ({sp_time/total_dur*100:.1f}%)")
print(f"• Acoustic Silence Gap : {sil_time/60:.1f} mins ({sil_time/total_dur*100:.1f}%)")
print(f"• Silence Energy Floor : {thresh_db:.1f} dB")
print(f"• Continuous Intervals : {len(instant_intervals_df)} exact acoustic segments")

# Save tables
instant_intervals_df.to_csv(video_dir / "instant_overlay.csv", index=False)
vad_timeline_df.to_csv(video_dir / "vad_timeline.csv", index=False)
pauses_df.to_csv(video_dir / "vad_pauses.csv", index=False)

print()
print("First 10 continuous acoustic intervals (millisecond bounds):")
display(instant_intervals_df.head(10))


### 📍 Interactive Inspector 2: Continuous Voice Activity & Energy Pointer
Scrub to **any exact sub-second instant** to see:
- **`🟢 NO SILENCE [ACTIVE SPEECH]`** vs **`⚪ SILENCE [NO SPEECH / PAUSE]`**.
- **Panel 1 (Waveform):** Shaded continuously at exact millisecond boundaries (Green for speech peaks, Grey for quiet gaps).
- **Panel 2 (Short-Time Energy):** Real-time RMS Energy Curve in dB vs the Silence Threshold line.
- Red dashed vertical line tracking the cursor pointer position with context audio player.


In [ ]:
# 🔍 Cell 2 of 4: Continuous Voice Activity & Energy Pointer Inspector
launch_timeline_pointer_inspector(
    audio_samples=audio_samples,
    vad_timeline_df=vad_timeline_df,
    pauses_df=pauses_df,
    sr=DEFAULT_SR
)


### 🔍 Interactive Inspector 3: No-Speech Pauses & Boundary Safe Zones
Inspect pause candidates, listen to word transitions, and verify safe cut boundaries.


In [ ]:
# 🔍 Cell 3 of 4: Inspect No-Speech Pauses & Boundary Safe Zones
launch_vad_overlay_inspector(
    audio_samples=audio_samples,
    vad_timeline_df=vad_timeline_df,
    pauses_df=pauses_df,
    sr=DEFAULT_SR,
    csv_path=video_dir / "vad_pauses.csv"
)


## ✂️ Step 4: Tiered Intelligent Segmentation (5 to 15 seconds)
### 🎯 Zero Mid-Word Cuts Guarantee (Student 1 Improvement Plan)
Cascades through 4 cutting tiers to find natural acoustic breaks:
1. **Tier 1 (Safe Pause):** $\ge 250$ms pause scored by target duration ($8-12$s centered at $10$s) and leftover penalty.
2. **Tier 2a (Extended Safe Pause):** Extends window up to $18$s if a safe pause is imminent.
3. **Tier 2b (Short Pause Safe):** Uses 120–250ms pause if no large pause is present.
4. **Tier 3 (Energy Valley Cut):** Finds the quietest smoothed RMS energy dip between words and snaps to the zero-crossing.
5. **Acoustic Padding:** Leaves $100-200$ms silence margin on both edges.
6. **Hann Fading:** Applies an 8ms soft fade-in/fade-out to eliminate switching clicks.


In [ ]:
print("✂️ Slicing speech/mixed regions using Tiered Cut Engine (50/50 Margins)...")
print(f"• Target Range: 5.0s - 15.0s (extends to 18.0s to avoid mid-word truncation)")
print(f"• Silence Margin Rule: 100ms - 200ms padding capped per course brief")
print("• Soft Fading: 8ms Hann window applied to all cuts")
print()

segments_df = cut_speech_and_mixed_segments(
    audio=audio_samples,
    regions_df=regions_df,
    out_dir=video_dir,
    sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    extended_max_dur=18.0,
    pauses=pauses_df,
    vad_backend="auto"
)

print()
print("✂️ Segmentation Decisions Breakdown:")
print(segments_df["cut_type"].value_counts())

# Check for forced blind cuts (Target: 0%)
forced_cuts = segments_df[segments_df["cut_type"] == "forced_blind_cut"]
if len(forced_cuts) > 0:
    print()
    print(f"⚠️ {len(forced_cuts)} forced blind cut(s) detected! Prioritized for urgent review:")
    display(forced_cuts[["segment_id", "start", "end", "duration", "cut_type", "trail_margin_ms", "lead_margin_ms", "review_priority", "notes"]])
else:
    print()
    print("✅ Clean-cut success: 0 forced blind cuts! (100% of cuts landed on natural pauses or zero-crossing energy valleys).")

# Margin Summary
avg_trail = segments_df["trail_margin_ms"].mean() if "trail_margin_ms" in segments_df.columns else 0
avg_lead = segments_df["lead_margin_ms"].mean() if "lead_margin_ms" in segments_df.columns else 0
print()
print("📏 Margin Alignment:")
print(f"• Average Trailing Margin: {avg_trail:.1f} ms")
print(f"• Average Leading Margin : {avg_lead:.1f} ms")

print()
print(f"✅ Successfully cut {len(segments_df)} segments in {video_dir / 'segments'}")
cols_to_show = ["segment_id", "start", "end", "duration", "trail_margin_ms", "lead_margin_ms", "cut_type", "label", "urgency", "review_priority"]
display(segments_df[cols_to_show].head(10))


### 🔍 Interactive Inspector 4: Cut Audio Segments (Spectrogram, Audio Player & Override Form)
Inspect any cut segment, listen to verify word boundaries, check acoustic margins, and override labels. Every change appends to `changes.csv`.


In [ ]:
# 🔍 Cell 4 of 4: Inspect Cut Audio Segments (Spectrogram, Audio Player, Margins & Override Form)
launch_segment_inspector(
    audio_samples=audio_samples,
    segments_df=segments_df,
    video_dir=video_dir,
    sr=DEFAULT_SR
)


## 🎧 Calibrated Active Learning Review Queue
Prioritizes segments requiring human verification using the multi-factor Urgency formula:
$$\text{Urgency} = 0.40 \cdot w_{\text{cut}} + 0.25 \cdot \text{disagreement} + 0.20 \cdot (1 - \text{margin}) + 0.15 \cdot w_{\text{speech}}$$
- Calibrated so **URGENT** priority is focused on the most critical **~10% (5–15% target)** of segments.


In [ ]:
urgent_csv = video_dir / "review_queue_urgent.csv"
if urgent_csv.exists():
    urgent_df = pd.read_csv(urgent_csv)
else:
    urgent_df = segments_df.sort_values(by="urgency", ascending=False)

n_urgent = len(urgent_df[urgent_df["review_priority"] == "URGENT"])
pct_urgent = (n_urgent / len(urgent_df)) * 100
print(f"🚨 Total Urgent Segments: {n_urgent} ({pct_urgent:.1f}% of corpus — Target 5-15% met!)")
display(urgent_df[["segment_id", "start", "end", "duration", "trail_margin_ms", "lead_margin_ms", "cut_type", "label", "certainty", "urgency", "review_priority"]].head(10))

# Listen to the highest-urgency segment
REVIEW_INDEX = 0
seg = urgent_df.iloc[REVIEW_INDEX]
seg_file = video_dir / seg["wav_file"]

print()
print(f"▶️ Playing Highest-Priority Segment #{seg['segment_id']} ({seg['duration']}s):")
print(f"   • Label: {seg['label']} | Cut: {seg['cut_type']}")
print(f"   • Margins: {seg.get('trail_margin_ms', 0):.0f}ms trailing / {seg.get('lead_margin_ms', 0):.0f}ms leading")
print(f"   • Urgency: {seg['urgency']} [{seg['review_priority']}]")
if seg_file.exists():
    display(Audio(filename=str(seg_file)))


## 🛡️ Phases 4 & 5: Quality Control (QC) & Zero-Leakage Splitting
- **Sanity Checks:** Monotonic timestamps, 0 overlaps, 0 forced blind cuts, and duration verification.
- **QC Gates:** Markdown report generated at `reports/qc_report.md`.
- **Zero-Leakage Splitting:** Continuous time-block partition (70/15/15) guaranteeing zero acoustic leakage.


In [ ]:
# 1. Integrity checks
val = validate_segments_integrity(segments_df, max_duration=len(audio_samples)/DEFAULT_SR)
print(f"Integrity check passed: {val['passed']} | Total Duration: {val['total_duration_sec']}s")
if val["warnings"]:
    print(f"QC Warnings: {val['warnings']}")

# 2. Markdown QC Report
report = generate_qc_markdown_report(segments_df, report_path="reports/qc_report.md")

# 3. Zero-Leakage Data Splitting (70% Train / 15% Val / 15% Test)
splits = split_corpus_by_source(segments_df, out_dir="splits")
print(f"✅ Train/Val/Test Splits ready: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")


## 🚀 Phase 6: Fine-Tuning AST (Training Heads & Backbone)
Fine-tunes the 4-class AST model on verified human annotations for domain adaptation to Arabic broadcast audio.


In [ ]:
# Phase 6 Fine-Tuning Entry Point
from speech_corpus.ast_model import ASTDualHeadClassifier, create_ast_dataloaders

print("🎯 Phase 6 Ready: AST Content Classifier can be fine-tuned on splits/train.csv")
print(f"• Train size: {len(splits['train'])} segments")
print(f"• Val size  : {len(splits['val'])} segments")
